# Two Traders with Private Signals

Two traders trade one asset. Each observes a private signal, its own inventory and the common price, and the price
drift depends on both signals and both trading rates.

- Signals: $d\alpha^i_t = -\kappa\,\alpha^i_t\,dt + \sigma_\alpha\,dW^i_t$, observed by trader $i$ only.
- Price: $dS_t = \mu_t\,dt + \sigma\,dW^S_t$, with $\mu_t = \alpha^1_t + \alpha^2_t + \lambda(\nu^1_t + \nu^2_t)$.
- Inventories: $dq^i_t = \nu^i_t\,dt$.
- Criterion: trader $i$ maximizes $\mathbb{E}\big[\int_0^T (q^i_t \mu_t - a(\nu^i_t)^2 - \phi(q^i_t)^2)\,dt - \psi(q^i_T)^2\big]$ over $\nu^i$ adapted to the filtration of $(\alpha^i, q^i, S)$.

Trader $i$ infers the other signal from the price, which also moves with the other trader's trades, and those trades
depend on the other trader's inference of $\alpha^i$. No finite-dimensional filter closes this hierarchy. The game is
solved on the Euler--Maruyama grid with $N = 50$ steps on $[0, 1]$.

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
import pig
from pig import Solver, relative_l2, use_style, LEARNED, LINEAR
use_style()
device = pig.use_device()
to_players = lambda controls: [torch.tensor(controls[:, i], dtype=torch.float32) for i in range(2)]
print('device:', device)

from pig.games import TwoTraders, LinearStrategies
game = TwoTraders(lam=1.0)

## 1. Exact solution in the linear class

In this Gaussian linear-quadratic game, a best response to a strategy linear in the observed path is itself linear.
The equilibrium therefore lies in the class
$\nu^i_{t_j} = \sum_{k \le j} \big(\beta^i_{jk,0}\,\alpha^i_{t_k} + \beta^i_{jk,1}\,S_{t_k}\big)$.
Every process is then linear in the Gaussian shocks of the discretised game, so each trader's cost is a quadratic
function of its coefficients, computed exactly without simulation. `LinearStrategies` iterates damped best responses
over the coefficients.

- Open loop: in each best response the other trader's control is held fixed as a process.
- Closed loop: the other trader's coefficients are held fixed, so its control responds to the price.

In [ ]:
exact_solver = LinearStrategies(game)
beta_open, beta_closed = exact_solver.solve(closed=False), exact_solver.solve(closed=True)
validation = game.sample(2048, 7)                                  # the validation paths of every Solver
exact_open, exact_closed = exact_solver.controls(beta_open, validation), exact_solver.controls(beta_closed, validation)
print(f'mean |nu|: open loop {np.abs(exact_open).mean():.4f}, closed loop {np.abs(exact_closed).mean():.4f}')
print(f'relative L2 distance between the two equilibria: {relative_l2(torch.tensor(exact_closed), torch.tensor(exact_open)):.3f}')

## 2. The learned open-loop equilibrium

The learned method computes the open-loop equilibrium by damped Picard iteration. Each trader's control is a network
of its own observations.

- Policy: a GRU with hidden state in $\mathbb{R}^{32}$ takes the scaled increments of $(\alpha^i, S)$ as input; a
  width-64 network maps $(t, \alpha^i_t, q^i_t, S_t)$ and the GRU state to $\nu^i_t$.
- Target: $\nu^i_t - (\partial J^i / \partial \nu^i_t) / (2a\,\Delta t)$, the derivative of the realised cost by automatic
  differentiation with the other trader's controls held fixed.
- Regression: minimizing a mean squared error between the policy and the damped target, with damping 0.6 and a
  martingale control variate subtracted from the target.
- Budget: 16384 paths, 750 Adam steps per iteration, 10 iterations, controls averaged from iteration 4.

In [ ]:
solver = Solver(game, damping=0.6, steps=750, batch=2048, n_paths=16384, control_variate=True)
learned, history = solver.run(10, average_from=4, reference=to_players(exact_open))
print(f'relative L2 error against the exact open-loop equilibrium: {relative_l2(learned[0], to_players(exact_open)[0]):.3f}, '
      f'{relative_l2(learned[1], to_players(exact_open)[1]):.3f}')

In [ ]:
def plot_controls(learned, exact, titles, linear=None, paths=(0, 1, 2), ylabel='control'):
    t = np.arange(learned[0].shape[1]) * game.dt
    fig, axes = plt.subplots(1, len(learned), figsize=(10, 3.2), sharey=True)
    for i, ax in enumerate(axes):
        for n, path in enumerate(paths):
            ax.plot(t, learned[i][path].cpu(), color=LEARNED, lw=2.6, alpha=0.45, label='learned' if n == 0 else None)
            if linear is not None:
                ax.plot(t, linear[i][path].cpu(), color=LINEAR, lw=1.0, label='linear strategies' if n == 0 else None)
            ax.plot(t, exact[i][path].cpu(), 'k--', lw=0.8, label='exact' if n == 0 else None)
        ax.set_xlabel('$t$'); ax.set_title(titles[i])
    axes[0].set_ylabel(ylabel); axes[0].legend()
    plt.show()

plot_controls(learned, to_players(exact_open), ['trader 1', 'trader 2'], ylabel=r'$\nu_t$')

## 3. Open and closed loop

At $\lambda = 1.5$ the open-loop and closed-loop equilibria differ by about 8.5%. Gradient play computes the
closed-loop one: each trader takes Adam steps on its own realised cost, differentiated through the simulated game
with the other trader's network inside, starting from the open-loop solution. The open-loop variant of the same
scheme, with the other trader's controls detached, is the control experiment.

In [ ]:
game = TwoTraders(lam=1.5); exact_solver = LinearStrategies(game)
exact = {loop: to_players(exact_solver.controls(exact_solver.solve(closed=loop == 'closed'), validation)) for loop in ('open', 'closed')}
results = {}
for loop in ('open', 'closed'):
    solver = Solver(game, damping=0.6, steps=750, batch=2048, n_paths=16384, control_variate=True)
    solver.run(6, average_from=3, log=lambda message: None)
    results[loop], _ = pig.gradient_play(solver, 10, steps=100, closed=loop == 'closed', log=lambda message: None)
for loop in ('open', 'closed'):
    print(f'gradient play, {loop} loop: error against the exact open loop {relative_l2(results[loop][0], exact["open"][0]):.3f}, '
          f'against the exact closed loop {relative_l2(results[loop][0], exact["closed"][0]):.3f} (trader 1)')